In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from sentence_transformers import SentenceTransformer

from src.data import load_dataset
from src.embedder import record_to_text, get_embeddings, save_embeddings, load_embeddings, get_cache_path
from src.search import search_all_questions
from src.metrics import evaluate
from src.config import MODELS, TOP_K

C:\Users\Admin\semantic-search-case\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
corpus, questions, categories = load_dataset()
corpus_texts = [record_to_text(item) for item in corpus]

print(f"Corpus: {len(corpus)} chunks")
print(f"Questions: {len(questions)}")


Corpus: 200 chunks
Questions: 25


In [3]:
model_name = MODELS[0]
print(f"Model: {model_name}")

Model: paraphrase-multilingual-MiniLM-L12-v2


In [4]:
cache_path = get_cache_path(model_name)

if Path(cache_path).exists():
    corpus_embedding = load_embeddings(cache_path)
    print("Loaded from cache")
else:
    corpus_embedding = get_embeddings(corpus_texts, model_name)
    save_embeddings(cache_path, corpus_embedding)
    print("Computed and saved to cache")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 7/7 [00:22<00:00,  3.21s/it]

Computed and saved to cache


In [5]:
model = SentenceTransformer(model_name)
search_outputs = search_all_questions(questions, model, corpus, corpus_embedding)

Loading weights: 100%|██████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 904.53it/s]


In [6]:
metrics = evaluate(search_outputs, k=TOP_K)
print(metrics)

{'precision_at_3': 0.84, 'mrr': 0.56, 'num_questions': 25}


In [7]:
print("Model name:", model_name)
print("TOP_K:", TOP_K)
print("MODELS list:", MODELS)

Model name: paraphrase-multilingual-MiniLM-L12-v2
TOP_K: 3
MODELS list: ['paraphrase-multilingual-MiniLM-L12-v2', 'paraphrase-multilingual-mpnet-base-v2', 'intfloat/multilingual-e5-small']
